# Hands-on: Training the AIFS with Anemoi

In this tutorial we will learn how to train the AIFS (deterministic) model using the Anemoi packages.

**Learning Objectives**

By the end of this tutorial, you will:
- Understand the setup for deterministic AIFS training
- Learn how to configure the anemoi training pipeline
- Build a minimal training configuration step-by-step
- Execute a short training run to verify everything works
- Explore advanced configurations such as "multi-dataset" or "stretched-grid"


**Resources**

- [Anemoi Documentation](https://anemoi.readthedocs.io/projects/training/en/latest/)
- [Lang et al. 2024](https://arxiv.org/abs/2406.01465)
- [Nipen et al. 2024](https://arxiv.org/abs/2409.02891)

## Step 1: Git clone the anemoi-core repo

```bash
git clone git@github.com:ecmwf/anemoi-core.git
```

***IMPORTANT NOTE***

The `main` branch of `anemoi-core` is under rapid development. The structures below might have slightly changed since I drafted this notebook. Always compare the structures described here against the examples given in the Anemoi-core repository.

## Building Our Training Config

Now we'll build a multi-dataset training configuration step-by-step. The aim is to build a minimal configuration file that we can use to train a dummy model on three datasets

```yaml
dataset_a: ea-o96-pl6-20240821-20240830-6h.zarr
dataset_b: ma-lm15-pl6-20240821-20240830-6h.zarr
dataset_c: observations-himawari-ahi-zarr2-o512-202408-6h-v1.zarr
```

| Key | Dataset | Description |
|-----|---------|-------------|
| `ea_o96_pl6` | `ea-o96-pl6-20240821-20240830-6h.zarr` | ERA5 reanalysis, O96 global, 6 pressure levels |
| `jma_lm15_pl6` | `ma-lm15-pl6-20240821-20240830-6h.zarr` | JMA high-resolution (15 km), 6 pressure levels |
| `himawari_obs` | `observations-himawari-ahi-zarr2-o512-202408-6h-v1.zarr` | Himawari-8/9 AHI satellite observations |

## How the config system works

Anemoi training uses **Hydra** for configuration. A top-level YAML file selects config groups via `defaults:`, then each group is a separate YAML in a sub-directory. The key config layers for multi-dataset training are:

```
config/
├── three_ds.yaml              # Top-level: selects defaults, hardware overrides
├── data/three_ds.yaml         # Per-dataset: forcing, diagnostics, preprocessors
├── dataloader/three_ds.yaml   # Per-dataset: splits, grid indices, batch composition
├── training/three_ds.yaml     # Per-dataset: loss, metrics, variable groups
└── training/scalers/three_ds.yaml  # Per-dataset: loss scaler weights
```

We will create these files during this hands-on exercise and test-run the configuration.

## Step 1: Top-Level Config

Create `training/src/anemoi/training/config/three_ds.yaml`:

```yaml
defaults:
- data: three_ds
- dataloader: three_ds
- diagnostics: evaluation_multi
- hardware: ...             # or your site-specific hardware config
- graph: multi_scale
- model: graphtransformer
- training: three_ds
- _self_

config_validation: False     # set True once your config is stable

training:
  model_task: anemoi.training.train.tasks.GraphForecaster
  max_steps: 200
  val_check_interval: 50
  lr:
    warmup: 10
    rate: 0.625e-4
    iterations: ${training.max_steps}
    min: 3.e-7
  model:
    num_channels: 512

data:
  resolution: o96
  frequency: 6h
  timestep: 6h

dataloader:
  limit_batches:
    training: null
    validation: null
    test: 20
  num_workers:
    training: 4
    validation: 4
    test: 1
  batch_size:
    training: 4
    validation: 4
    test: 2

hardware:
  paths:
    data: ...  # the data directory
    output: ${oc.env:SCRATCH}/anemoi-test/multi-dataset/
  files:
    graph: graph_anemoi_new_${data.resolution}.pt

    # One entry per dataset — these are referenced in the dataloader config
    dataset_a: ea-o96-pl6-20240821-20240830-6h.zarr
    dataset_b: ma-lm15-pl6-20240821-20240830-6h.zarr
    dataset_c: observations-himawari-ahi-zarr2-o512-202408-6h-v1.zarr

  num_gpus_per_model: 1

diagnostics:
  enable_checkpointing: False
  log:
    mlflow:
      enabled: False
    wandb:
      enabled: False
    interval: 50
```

**Key points:**
- Each dataset gets a `hardware.files.dataset_*` entry so the dataloader can reference it via `${hardware.files.dataset_a}`, etc.
- `config_validation: False` is useful while iterating; switch to `True` once stable.
- The `defaults:` list points to the three config groups we create next.

## Step 2: Data Config — Forcing, Diagnostics, Processors

Create `training/src/anemoi/training/config/data/three_ds.yaml`. Start from this example:

```yaml
# Multi-dataset data configuration for debugging with era5 and era5_copy
format: zarr
# Time frequency requested from dataset
frequency: 6h
# Time step of model (must be multiple of frequency)
timestep: 6h

# Dataset-specific configurations
datasets:
  ea_o96_pl6:
    forcing:
    - "cos_latitude"
    - "cos_longitude"
    - "sin_latitude"
    - "sin_longitude"
    - "cos_julian_day"
    - "cos_local_time"
    - "sin_julian_day"
    - "sin_local_time"
    - "insolation"
    - "lsm"
    - "orog"
    diagnostic: [tp]
    processors:
      normalizer:
        _target_: anemoi.models.preprocessing.normalizer.InputNormalizer
        config:
          default: "mean-std"
          std:
          - "tp"
          min-max:
          max:
          - "orog"
          - "tcc"
          none:
          - "cos_latitude"
          - "cos_longitude"
          - "sin_latitude"
          - "sin_longitude"
          - "cos_julian_day"
          - "cos_local_time"
          - "sin_julian_day"
          - "sin_local_time"
          - "insolation"
          - "lsm"

  jma_lm15_pl6:
    forcing:
    - "cos_latitude"
    - "cos_longitude"
    - "sin_latitude"
    - "sin_longitude"
    - "cos_julian_day"
    - "cos_local_time"
    - "sin_julian_day"
    - "sin_local_time"
    - "lsm"
    - "orog"
    diagnostic: [tp]
    processors:
      normalizer:
        _target_: anemoi.models.preprocessing.normalizer.InputNormalizer
        config:
          default: "mean-std"
          std:
          - "tp"
          min-max:
          max:
          - "orog"
          - "tcc"
          none:
          - "cos_latitude"
          - "cos_longitude"
          - "sin_latitude"
          - "sin_longitude"
          - "cos_julian_day"
          - "cos_local_time"
          - "sin_julian_day"
          - "sin_local_time"
      const_imputer:
        _target_: anemoi.models.preprocessing.imputer.InputImputer
        config:
          default: "mean"

# Values set in the code
num_features: null # number of features in the forecast state
```

### Exercise
Add a block for the Himawari observations. Make sure the key matches that in the top-level config! 

**Hint**: Inspect the dataset using anemoi-dataset to see the variables and the dates it contains.

```yaml
  # ──────────────── Himawari satellite observations ────────────────
  himawari_obs:
    forcing:
    # TODO: Replace with actual forcing variables available in the Himawari dataset.
    - ...
    # TODO: List diagnostic-only variables (predicted but not fed back as input)
    diagnostic: []

    processors:
      normalizer:
        _target_: anemoi.models.preprocessing.normalizer.InputNormalizer
        config:
          default: "mean-std"
          std:
          min-max:
          max:
          none:
          - ...  # TODO
      const_imputer:
        _target_: anemoi.models.preprocessing.imputer.InputImputer
        config:
          default: "mean"
          # Satellite obs often have missing values (e.g. cloud-masked pixels).
          # "mean" imputation fills NaNs with the training-set mean.

num_features: null
```



**Key points:**
- Each dataset key (e.g. `ea_o96_pl6`) must be used consistently across **all** config files.
- The `forcing` list defines variables that are inputs but not predicted. The `diagnostic` list defines variables that are predicted but not fed back.
- Datasets with missing grid points (JMA, satellite obs) need a `const_imputer` processor.
- The Himawari section uses `TODO` placeholders — fill these in once you inspect the dataset's variable list (e.g. via `anemoi-datasets inspect`).

## Step 3: Dataloader Config — Splits, Grid Indices, Batch Composition

Create `training/src/anemoi/training/config/dataloader/three_ds.yaml` by adapting `training/src/anemoi/training/config/dataloader/multi.yaml`:

```yaml
prefetch_factor: 2
pin_memory: True

read_group_size: ${hardware.num_gpus_per_model}

num_workers:
  training: 1
  validation: 1
  test: 1
batch_size:
  training: 2
  validation: 2
  test: 2

limit_batches:
  training: null
  validation: null
  test: 20

# ============
# Grid indices: define how each dataset maps to graph nodes
# ============
grid_indices:
  datasets:
    ea_o96_pl6:
      _target_: anemoi.training.data.grid_indices.FullGrid
      nodes_name: ${graph.data}
    jma_lm15_pl6:
      _target_: anemoi.training.data.grid_indices.FullGrid
      nodes_name: ${graph.data}
    himawari_obs:
      _target_: anemoi.training.data.grid_indices.FullGrid
      nodes_name: ${graph.data}

# ============
# Multi-dataset split definitions
# Each batch is a dict: {"ea_o96_pl6": tensor, "jma_lm15_pl6": tensor, "himawari_obs": tensor}
# CONSTRAINT: All datasets must have the same number of valid time samples per split.
# ============

training:
  datasets:
    ea_o96_pl6:
      dataset: ${hardware.files.dataset_a}
      start: 2024-08-21
      end: 2024-08-31
      frequency: ${data.frequency}
      drop: []
      trajectory: null
    jma_lm15_pl6:
      dataset: ${hardware.files.dataset_b}
      start: 2024-08-21
      end: 2024-08-31
      frequency: ${data.frequency}
      drop: []
      trajectory: null
    himawari_obs:
      dataset: ${hardware.files.dataset_c}
      start: 2024-08-21
      end: 2024-08-31
      frequency: ${data.frequency}
      drop: []
      trajectory: null

validation_rollout: 1

validation: ... # same datasets and dates

test: ... # same datasets and dates
```

**Key points:**
- Every dataset key must match exactly between `data/`, `dataloader/`, and `training/` configs.
- All datasets must produce the **same number of valid timesteps** per split. Align `start`/`end`/`frequency` carefully.
- The `grid_indices` section maps each dataset to graph nodes. If a dataset covers only a regional domain, you may need a specialized grid indices class instead of `FullGrid`.


## Step 4: Training Config — Loss, Metrics, Scalers

Create `training/src/anemoi/training/config/training/three_ds.yaml`.

Start from the existing `training/multi.yaml` and add a third dataset block everywhere:

```yaml
# ... first bits are the same as those in multi.yaml ...

# ============
# Per-dataset training loss
# ============
training_loss:
  datasets:
    ea_o96_pl6:
      _target_: anemoi.training.losses.MSELoss
      scalers: ['pressure_level', 'general_variable', 'nan_mask_weights', 'node_weights', 'time_steps']
      ignore_nans: false
    jma_lm15_pl6:
      _target_: anemoi.training.losses.MSELoss
      scalers: ['pressure_level', 'general_variable', 'nan_mask_weights', 'node_weights', 'time_steps']
      ignore_nans: false
    himawari_obs:
      _target_: anemoi.training.losses.MSELoss
      scalers: ['general_variable', 'nan_mask_weights', 'node_weights', 'time_steps']
      # NOTE: If satellite obs have no pressure levels, omit 'pressure_level' scaler.
      ignore_nans: true
      # ignore_nans: true is important for satellite obs which may have masked pixels.

# ============
# Per-dataset validation metrics
# ============
validation_metrics:
  datasets:
    ea_o96_pl6:
      mse:
        _target_: anemoi.training.losses.MSELoss
        scalers: ['node_weights', 'time_steps']
        ignore_nans: false
    jma_lm15_pl6:
      mse:
        _target_: anemoi.training.losses.MSELoss
        scalers: ['node_weights', 'time_steps']
        ignore_nans: false
    himawari_obs:
      mse:
        _target_: anemoi.training.losses.MSELoss
        scalers: ['node_weights', 'time_steps']
        ignore_nans: true

# ============
# Per-dataset variable groups (for variable-level scaling)
# ============
variable_groups:
  datasets:
    ea_o96_pl6:
      default: sfc
      pl:
        param: [q, t, u, v, w]
      sfc:
        param: [tp, cp, 10u, 10v, 2d, sp]
    jma_lm15_pl6:
      default: sfc
      pl:
        param: [q, t, u, v, w]
      sfc:
        param: [tp, cp, 10u, 10v, 2d]
    himawari_obs:
      default: obs
      # TODO: Define variable groups based on actual Himawari channels/variables.
      # Example for brightness temperature channels:
      # sfc:
      #   param: [bt_channel2, ...]

# ============
# Per-dataset tracked metrics
# ============
metrics:
  datasets:
    ea_o96_pl6:
    - z_500
    - z_1000
    - t_500
    - u_850
    - v_850
    jma_lm15_pl6:
    - z_500
    - z_1000
    - t_500
    - u_850
    himawari_obs: []
    # TODO: Add satellite-specific metrics once the variables are known.
```

### Scalers

Create `training/src/anemoi/training/config/training/scalers/three_ds.yaml`:

```yaml
datasets:
  # ──────────────── ERA5 scalers ────────────────
  ea_o96_pl6:
    general_variable:
      _target_: anemoi.training.losses.scalers.GeneralVariableLossScaler
      weights:
        default: 1
        q: 0.8
        t: 6
        u: 0.8
        v: 0.5
        w: 0.001
        z: 12
        sp: 10
        10u: 0.1
        10v: 0.1
        2d: 0.5
        tp: 0.025
        cp: 0.0025
    nan_mask_weights:
      _target_: anemoi.training.losses.scalers.NaNMaskScaler
    pressure_level:
      _target_: anemoi.training.losses.scalers.ReluVariableLevelScaler
      group: pl
      y_intercept: 0.2
      slope: 0.001
    stdev_tendency:
      _target_: anemoi.training.losses.scalers.StdevTendencyScaler
    var_tendency:
      _target_: anemoi.training.losses.scalers.VarTendencyScaler
    node_weights:
      _target_: anemoi.training.losses.scalers.GraphNodeAttributeScaler
      nodes_name: ${graph.data}
      nodes_attribute_name: area_weight
      norm: unit-sum
    time_steps:
      _target_: anemoi.training.losses.scalers.UniformTimeStepScaler
      multistep_output: ${training.multistep_output}

  # ──────────────── JMA scalers ────────────────
  jma_lm15_pl6:
    general_variable:
      _target_: anemoi.training.losses.scalers.GeneralVariableLossScaler
      weights:
        default: 1
        q: 1.0
        t: 8
        u: 0.8
        v: 0.5
        w: 0.001
        z: 3
    nan_mask_weights:
      _target_: anemoi.training.losses.scalers.NaNMaskScaler
    pressure_level:
      _target_: anemoi.training.losses.scalers.ReluVariableLevelScaler
      group: pl
      y_intercept: 0.1
      slope: 0.001
    stdev_tendency:
      _target_: anemoi.training.losses.scalers.StdevTendencyScaler
    var_tendency:
      _target_: anemoi.training.losses.scalers.VarTendencyScaler
    node_weights:
      _target_: anemoi.training.losses.scalers.GraphNodeAttributeScaler
      nodes_name: ${graph.data}
      nodes_attribute_name: area_weight
      norm: unit-sum
    time_steps:
      _target_: anemoi.training.losses.scalers.UniformTimeStepScaler
      multistep_output: ${training.multistep_output}

  # ──────────────── Himawari scalers ────────────────
  himawari_obs:
    general_variable:
      _target_: anemoi.training.losses.scalers.GeneralVariableLossScaler
      weights:
        default: 1
        # TODO: You can set per-variable weights once the Himawari variable list is known.
    nan_mask_weights:
      _target_: anemoi.training.losses.scalers.NaNMaskScaler
    # No pressure_level scaler — satellite obs are typically "single-level".
```

**Key points:**
- Every scaler name referenced in `training_loss.datasets.<key>.scalers` must be defined in the corresponding scalers file.
- Satellite observations typically do not have pressure levels, so `pressure_level` is omitted from the Himawari scalers and loss config.
- The `nan_mask_weights` scaler is important for any dataset with missing values — it zeros out imputed grid points in the loss.


## Launch Training

### Local (single GPU)

```bash
cd /path/to/anemoi-core
anemoi-training train --config-name=three_ds
```

### Useful Hydra Overrides

```bash
# Limit batches for a quick smoke test
anemoi-training train --config-name=three_ds \
  dataloader.limit_batches.training=10 \
  dataloader.limit_batches.validation=5 \
  training.max_steps=20

# Override a single dataset path
anemoi-training train --config-name=three_ds \
  hardware.files.dataset_c=/new/path/to/himawari.zarr
```

## Troubleshooting

### Config validation errors

Set `config_validation: False` in the top-level config while iterating. Re-enable once the config structure is stable. Pydantic schema errors will show which field is invalid.

### "Datasets have different number of valid samples"

All datasets in a multi-dataset setup must produce the same number of timesteps for each split. Check that `start`, `end`, and `frequency` align across all three datasets in the dataloader config. Use `drop: [<timestamp>]` to exclude problematic timesteps.

### Missing variables

If a variable listed in `forcing` or `diagnostic` doesn't exist in the Zarr dataset, the dataloader will fail at startup. Inspect available variables with:
```bash
python -c "from anemoi.datasets import open_dataset; ds = open_dataset('/path/to/dataset.zarr'); print(ds.variables); print(ds.dates)"
```

### Useful environment variables

| Variable | Purpose |
|----------|---------|
| `HYDRA_FULL_ERROR=1` | Show full Hydra stack traces |
| `OC_CAUSE=1` | Show OmegaConf interpolation errors |
| `PYTHONFAULTHANDLER=1` | Print Python traceback on segfault |
| `NCCL_DEBUG=INFO` | Debug multi-GPU communication |
| `ANEMOI_BASE_SEED=42` | Fix random seed for reproducibility |

---

## Summary: Files to Create

| File | Based on |
|------|----------|
| `config/three_ds.yaml` | `config/multi_jma.yaml` |
| `config/data/three_ds.yaml` | `config/data/multi.yaml` |
| `config/dataloader/three_ds.yaml` | `config/dataloader/multi.yaml` |
| `config/training/three_ds.yaml` | `config/training/multi.yaml` |
| `config/training/scalers/three_ds.yaml` | `config/training/scalers/multi.yaml` |

All paths are relative to `training/src/anemoi/training/`.

# Extra credit: A more realistic training setup

Let us attempt to define and train a model on a much larger, o96 (ca. 1-degree) ERA5 dataset. The data can be downloaded from:

https://anemoi.readthedocs.io/projects/training/en/latest/user-guide/download-era5-o96.html

Once you have downloaded your data to your computer system, change the training configuration paths to point to that dataset. For example, in the top-level config:

```yaml
system:
  input:
    graph: graph_anemoi_new_${data.resolution}.pt

    # Primary dataset for ERA5
    dataset: era5-o96-1979-2023-6h-v8.zarr

  output:
    root: ${oc.env:SCRATCH}/anemoi-test/o96/
```